In [3]:
import importlib.metadata
import os
from dotenv import load_dotenv

load_dotenv()   # read GROQ_API_KEY, RAPIDAPI_KEY, OPIK_API_KEY from .env

# Check that all packages are installed
packages = ["langgraph", "langchain-core", "langchain-groq", "fastembed", "pymupdf", "opik"]

for package_name in packages:
    try:
        version = importlib.metadata.version(package_name)
        print(package_name, "→", version)
    except importlib.metadata.PackageNotFoundError:
        print(package_name, "→ NOT INSTALLED ← run pip install -r requirements.txt")

print()

# Check that API keys are set
# (Scoring uses a LOCAL embedding model via fastembed — no extra key needed.)
key_names = ["GROQ_API_KEY", "RAPIDAPI_KEY", "OPIK_API_KEY"]

for key_name in key_names:
    value = os.getenv(key_name, "")
    if value == "" or value.startswith("your_"):
        print(key_name, "→ MISSING — add it to your .env file")
    else:
        print(key_name, "→ OK")

langgraph → 1.2.10
langchain-core → 1.5.3
langchain-groq → 1.1.3
fastembed → 0.8.0
pymupdf → 1.28.0
opik → 2.2.13

GROQ_API_KEY → OK
RAPIDAPI_KEY → OK
OPIK_API_KEY → OK


In [4]:
import fitz  # PyMuPDF

def extract_text(pdf_path):
    all_text = ""
    pdf_file = fitz.open(pdf_path)

    for page in pdf_file:
        page_text = page.get_text()
        all_text += page_text

    pdf_file.close()
    return all_text


# Path to your local PDF
pdf_path = r"C:\Users\Asus\Downloads\KedarlingKanade_DevOps__Cloud.pdf"

# Extract text
cv_text = extract_text(pdf_path)

print("CV text length:", len(cv_text), "characters")
print("\nFirst 200 characters:")
print(cv_text[:200])

CV text length: 5447 characters

First 200 characters:
KEDARLING KANADE
 +91 76195 77838
# kedarkanade9417@gmail.com
+ Chikodi, Belagavi, Karnataka
ï linkedin.com/in/kedarling-kanade-7070b2232
§ github.com/Kedarling7838
Professional Summary
Aspiring DevO


In [7]:
from pydantic import BaseModel            # lets us define a class with typed fields
from langchain.chat_models import init_chat_model

# ── Define what fields we want from the CV ────────────────────────────────────
# BaseModel is from Pydantic — it gives us automatic type checking
class CVProfile(BaseModel):
    name:               str        # full name of the candidate
    current_title:      str        # their current job title
    skills:             list[str]  # list of individual skills
    years_experience:   int        # total years of work experience
    preferred_location: str        # where they want to work
    education:          str        # highest degree
    summary:            str        # 1-2 sentence description for job matching


# Create the LLM
llm = init_chat_model("groq:llama-3.3-70b-versatile")


def parse_cv(cv_text):
    # Write the prompt — give the LLM the full CV text
    prompt = """
Extract the candidate profile from this CV.

Return:
- name
- current_title
- skills as a list
- years_experience as an integer (example: 1, 2, 5)
- preferred_location
- education
- summary

CV:
""" + cv_text

    # with_structured_output tells the LLM: respond ONLY with JSON matching CVProfile
    parser  = llm.with_structured_output(CVProfile)
    profile = parser.invoke(prompt)   # send the prompt, get back a CVProfile object
    return profile


# ── Run it ────────────────────────────────────────────────────────────────────
cv_profile = parse_cv(cv_text)

print("CV Profile extracted:")
print("  Name        :", cv_profile.name)
print("  Title       :", cv_profile.current_title)
print("  Skills      :", cv_profile.skills)
print("  Experience  :", cv_profile.years_experience, "years")
print("  Location    :", cv_profile.preferred_location)
print("  Education   :", cv_profile.education)
print()
print("The LLM read the raw text and filled in the typed fields automatically.")
print("No regex, no manual parsing — structured output does it all.")

CV Profile extracted:
  Name        : KEDARLING KANADE
  Title       : Aspiring DevOps and Cloud Engineer
  Skills      : ['Python', 'Core Java', 'Bash', 'Shell Scripting', 'Amazon Web Services (AWS)', 'Microsoft Azure', 'Git', 'GitHub', 'Docker', 'Kubernetes', 'Terraform', 'Jenkins', 'Maven', 'Prometheus', 'Grafana', 'Splunk', 'Nginx', 'Apache Tomcat', 'Linux', 'Windows', 'MySQL', 'Oracle Database', 'Visual Studio Code', 'Postman', 'Git Bash', 'Jupyter Notebook', 'PuTTY', 'WinSCP']
  Experience  : 1 years
  Location    : Chikodi, Belagavi, Karnataka
  Education   : Master of Computer Applications (MCA) from GM University, Bachelor of Computer Applications (BCA) from Rani Channamma University

The LLM read the raw text and filled in the typed fields automatically.
No regex, no manual parsing — structured output does it all.


In [8]:
import os
import json
import requests
from langchain_core.tools import tool     # the @tool decorator

RAPIDAPI_KEY = os.getenv("RAPIDAPI_KEY", "")
print("API key found:", bool(RAPIDAPI_KEY))


# The @tool decorator turns this function into a tool the LLM is allowed to call.
@tool
def search_jobs(query, location="remote"):
    """Search for job openings matching a query and location.

    Args:
        query:    Job title or skills, e.g. 'Python data engineer'
        location: City, country, or 'remote'

    Returns:
        JSON string of up to 5 job listings.
    """
    url = "https://jsearch.p.rapidapi.com/search-v2"
    headers = {
        "x-rapidapi-key":  RAPIDAPI_KEY,
        "x-rapidapi-host": "jsearch.p.rapidapi.com",
    }
    params = {"query": query + " jobs in " + location, "num_pages": "1"}

    # Ask the API for jobs. If anything goes wrong, we fall back below.
    try:
        response = requests.get(url, headers=headers, params=params, timeout=30)
        data = response.json().get("data", [])
        jobs = data.get("jobs", []) if isinstance(data, dict) else data
    except Exception as err:
        print("Live API failed, using sample jobs:", err)
        jobs = []

    # No live jobs? Use the bundled sample_jobs.json so the demo always works.
    if not jobs:
        with open("sample_jobs.json", encoding="utf-8") as f:
            jobs = json.load(f)

    # Keep only the first 5 jobs and just the fields we care about.
    slim_jobs = []
    for job in jobs[:5]:
        slim_jobs.append({
            "job_title":       job.get("job_title", ""),
            "employer_name":   job.get("employer_name", ""),
            "job_description": (job.get("job_description") or "")[:600],
            "job_apply_link":  job.get("job_apply_link", ""),
            "job_country":     job.get("job_country", ""),
        })

    # Tools must return a string, so convert the list to a JSON string.
    return json.dumps(slim_jobs)


print("search_jobs tool ready. Tool name:", search_jobs.name)
print("Tool args:", list(search_jobs.args.keys()))

API key found: True
search_jobs tool ready. Tool name: search_jobs
Tool args: ['query', 'location']


In [10]:
# Call the tool directly (same as the graph will do internally)
try:
    raw_result = search_jobs.invoke({"query": "data engineer Python", "location": "remote"})

    # raw_result is a JSON string — convert it to a Python list
    jobs_found = json.loads(raw_result)

    print("Jobs returned:", len(jobs_found))
    print()

    for i, job in enumerate(jobs_found):
        number   = str(i + 1)
        title    = job["job_title"]
        employer = job["employer_name"]
        country  = job["job_country"]
        snippet  = job["job_description"][:120]
        print(number + ". " + title + " @ " + employer + " (" + country + ")")
        print("   " + snippet + "...")
        print()

    print("These are real live job listings from JSearch.")
    print("The agent will call this tool 2-3 times with different queries.")

except Exception as e:
    print("Error:", e)

Jobs returned: 5

1. Machine Learning Engineer @ Fractal Analytics (IN)
   We are looking for a Machine Learning Engineer to build and deploy ML models at scale. You will work with Python, PyTorc...

2. Computer Vision Engineer @ Tata Consultancy Services (IN)
   Join our AI research team to develop computer vision solutions covering object detection, image and video segmentation, ...

3. Generative AI Developer @ Infosys (IN)
   We are hiring a Generative AI Developer to build agentic AI applications using LLMs. Responsibilities include designing ...

4. NLP Engineer (Multilingual) @ Zoho Corporation (IN)
   Looking for an NLP Engineer to work on multilingual language understanding, ASR, speaker diarization and speech translat...

5. Data Scientist @ Mu Sigma (IN)
   As a Data Scientist you will turn raw data into business insight using Python, SQL and statistical modelling. You will b...

These are real live job listings from JSearch.
The agent will call this tool 2-3 times with diff

In [11]:
# We score jobs with EMBEDDINGS: turn the CV and each job into a list of numbers
# that captures its meaning, then measure how aligned they are (cosine similarity).
# fastembed runs a small model locally — free, no API key, no PyTorch crash.
import json
import numpy as np
from fastembed import TextEmbedding
from langchain_core.messages import ToolMessage

model = TextEmbedding("BAAI/bge-small-en-v1.5")   # first run downloads ~130 MB, then cached


def score_jobs(cv_profile, messages):
    # 1. Collect every job the agent found (each search result is a ToolMessage).
    jobs = []
    for message in messages:
        if isinstance(message, ToolMessage):
            jobs += json.loads(message.content)
    if not jobs:
        return []

    # 2. One line of text for the CV, and one for each job.
    cv_text   = cv_profile["current_title"] + " " + " ".join(cv_profile["skills"])
    job_texts = [job["job_title"] + " " + job["job_description"] for job in jobs]

    # 3. Turn every text into an embedding (a list of numbers = its meaning).
    vectors     = list(model.embed([cv_text] + job_texts))
    cv_vector   = vectors[0]
    job_vectors = vectors[1:]

    # 4. Score each job by how close its meaning is to the CV (cosine similarity, 0-100%).
    for job, vector in zip(jobs, job_vectors):
        similarity = np.dot(cv_vector, vector) / (np.linalg.norm(cv_vector) * np.linalg.norm(vector))
        job["match_score"] = round(float(similarity) * 100, 1)

    # 5. Best matches first — keep the top 5.
    jobs.sort(key=lambda job: job["match_score"], reverse=True)
    return jobs[:5]


# ── Test it ──────────────────────────────────────────────────────────────────
if "raw_result" not in globals():
    raw_result = search_jobs.invoke({"query": "data engineer Python", "location": "remote"})

test_message = ToolMessage(content=raw_result, tool_call_id="test")
scored = score_jobs(cv_profile.model_dump(), [test_message])

print("Scored", len(scored), "jobs:\n")
for job in scored:
    print(str(job["match_score"]) + "%  ->  " + job["job_title"] + " @ " + job["employer_name"])

d:\My Projects\Langgraph_job_search\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
d:\My Projects\Langgraph_job_search\rag\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\AppData\Local\Temp\fastembed_cache\models--qdrant--bge-small-en-v1.5-onnx-q. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more

Scored 5 jobs:

66.1%  ->  Generative AI Developer @ Infosys
63.6%  ->  Machine Learning Engineer @ Fractal Analytics
58.7%  ->  Computer Vision Engineer @ Tata Consultancy Services
58.3%  ->  NLP Engineer (Multilingual) @ Zoho Corporation
55.5%  ->  Data Scientist @ Mu Sigma


In [12]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages:    Annotated[list[AnyMessage], add_messages]  # appends each new message
    cv_text:     str    # the raw text from the PDF — set once at the start
    cv_profile:  dict   # the structured profile — set by parse_cv node
    scored_jobs: list   # the ranked jobs — set by score node


print("State has 4 fields:")
print()
print("  messages    — uses add_messages → new messages are APPENDED to the list")
print("  cv_text     — plain string → each node return REPLACES the value")
print("  cv_profile  — plain dict   → each node return REPLACES the value")
print("  scored_jobs — plain list   → each node return REPLACES the value")
print()
print("Why does messages need add_messages?")
print("Without it, returning {'messages': [new_msg]} would WIPE the entire history.")
print("add_messages makes it APPEND instead — so history is never lost.")

State has 4 fields:

  messages    — uses add_messages → new messages are APPENDED to the list
  cv_text     — plain string → each node return REPLACES the value
  cv_profile  — plain dict   → each node return REPLACES the value
  scored_jobs — plain list   → each node return REPLACES the value

Why does messages need add_messages?
Without it, returning {'messages': [new_msg]} would WIPE the entire history.
add_messages makes it APPEND instead — so history is never lost.


In [13]:
from langchain_core.messages import HumanMessage, SystemMessage

# Bind the tool to the LLM — same as Class 1
tools          = [search_jobs]
llm_with_tools = llm.bind_tools(tools)


# ── Node 1: parse_cv_node ─────────────────────────────────────────────────────
def parse_cv_node(state):
    profile      = parse_cv(state["cv_text"])  # LLM reads raw text, returns CVProfile
    profile_dict = profile.model_dump()         # convert Pydantic object to plain dict
    return {"cv_profile": profile_dict}


# ── Node 2: agent_node ────────────────────────────────────────────────────────
def agent_node(state):
    profile = state["cv_profile"]

    # Convert profile dict to readable text
    profile_text = json.dumps(profile, indent=2)

    # SystemMessage = background instructions given to the LLM
    # We tell it about the candidate so its searches are personalised
    system_message = SystemMessage(content=(
        "You are a job search assistant.\n"
        "Candidate profile:\n" + profile_text + "\n\n"
        "Make 2-3 targeted job searches using the search_jobs tool."
    ))

    # Combine system instruction + full conversation history
    all_messages = [system_message] + state["messages"]

    # Ask the LLM: call a tool or give final answer?
    response = llm_with_tools.invoke(all_messages)
    return {"messages": [response]}


# ── Routing function (same pattern as Class 1) ────────────────────────────────
def should_continue(state):
    last_message = state["messages"][-1]   # the most recent message
    if last_message.tool_calls:
        return "tools"   # LLM wants to search → go to tools node
    else:
        return "score"   # LLM is done → go to score node


# ── Node 3: score_node ────────────────────────────────────────────────────────
def score_node(state):
    scored = score_jobs(state["cv_profile"], state["messages"])
    return {"scored_jobs": scored}


# ── Node 4: format_node ───────────────────────────────────────────────────────
def format_node(state):
    jobs    = state["scored_jobs"]
    profile = state["cv_profile"]

    # Build a text block listing each job — one job per block
    job_lines = []
    for i, job in enumerate(jobs):
        number      = str(i + 1)
        title       = job.get("job_title", "")
        employer    = job.get("employer_name", "")
        score       = str(job.get("match_score", ""))
        description = job.get("job_description", "")[:300]
        apply_link  = job.get("job_apply_link", "N/A")

        line = number + ". " + title + " at " + employer + " [match: " + score + "%]"
        line = line + "\n   " + description
        line = line + "\n   Apply: " + apply_link
        job_lines.append(line)

    jobs_text = "\n\n".join(job_lines)   # put a blank line between each job

    skills_list = profile.get("skills", [])
    skills_text = ", ".join(skills_list)

    # Ask the LLM to write an encouraging personalised summary
    prompt = (
        "You are helping " + profile.get("name", "the candidate") + " find a job.\n"
        "Their title: " + profile.get("current_title", "") + "\n"
        "Skills: " + skills_text + "\n\n"
        "Top matched jobs:\n" + jobs_text + "\n\n"
        "Write a short encouraging summary of why these are good fits."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"messages": [response]}


print("Four nodes defined:")
print("  parse_cv_node  — reads cv_text, writes cv_profile")
print("  agent_node     — reads messages + cv_profile, writes messages")
print("  score_node     — reads messages, writes scored_jobs")
print("  format_node    — reads cv_profile + scored_jobs, writes final answer message")

Four nodes defined:
  parse_cv_node  — reads cv_text, writes cv_profile
  agent_node     — reads messages + cv_profile, writes messages
  score_node     — reads messages, writes scored_jobs
  format_node    — reads cv_profile + scored_jobs, writes final answer message


In [14]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

def build_graph():
    graph = StateGraph(State)   # tell LangGraph which State class to use

    # Add nodes
    graph.add_node("parse_cv", parse_cv_node)
    graph.add_node("agent",    agent_node)
    graph.add_node("tools",    ToolNode(tools))   # prebuilt node — runs any tool automatically
    graph.add_node("score",    score_node)
    graph.add_node("format",   format_node)

    # Add edges
    graph.add_edge(START,       "parse_cv")   # start here
    graph.add_edge("parse_cv",  "agent")      # parsed → start searching
    graph.add_conditional_edges("agent", should_continue)  # loop or move to score
    graph.add_edge("tools",     "agent")      # tool ran → back to agent
    graph.add_edge("score",     "format")     # scored → write summary
    graph.add_edge("format",    END)          # done

    return graph.compile()


app = build_graph()

print("Graph compiled successfully.")
print()
print("Class 1 graph:  START → agent ↔ tools → END")
print("Class 2 graph:  START → parse_cv → agent ↔ tools → score → format → END")
print()
print("The ReAct loop (agent ↔ tools) is the same as Class 1.")
print("The new parts are parse_cv before it and score + format after it.")

Graph compiled successfully.

Class 1 graph:  START → agent ↔ tools → END
Class 2 graph:  START → parse_cv → agent ↔ tools → score → format → END

The ReAct loop (agent ↔ tools) is the same as Class 1.
The new parts are parse_cv before it and score + format after it.


In [16]:
# The starting state — only cv_text is set, everything else starts empty
starting_state = {
    "cv_text":     cv_text,
    "messages":    [],
    "cv_profile":  {},
    "scored_jobs": [],
}

# Run the agent — simple invoke, no tracing.
# (Opik tracing is left out here on purpose so the notebook stays focused on the
#  core idea: parse the CV -> search -> score -> summarise. We demo Opik in the
#  Streamlit app instead.)
result = app.invoke(starting_state)

# The final answer is the last message
final_answer = result["messages"][-1].content
scored_jobs  = result.get("scored_jobs", [])

print("Messages in conversation :", len(result["messages"]))
print("Jobs scored              :", len(scored_jobs))
print()

print("Top matched jobs:")
for job in scored_jobs:
    score    = str(job.get("match_score", 0))
    title    = job.get("job_title", "")
    employer = job.get("employer_name", "")
    print("  " + score + "%  ->  " + title + " @ " + employer)

print()
print("Agent summary:")
print(final_answer)

Messages in conversation : 14
Jobs scored              : 5

Top matched jobs:
  72.5%  ->  Machine Learning Engineer @ Fractal Analytics
  72.5%  ->  Machine Learning Engineer @ Fractal Analytics
  72.5%  ->  Machine Learning Engineer @ Fractal Analytics
  72.5%  ->  Machine Learning Engineer @ Fractal Analytics
  72.5%  ->  Machine Learning Engineer @ Fractal Analytics

Agent summary:
KEDARLING KANADE, I'm excited to see that you have multiple job matches for Machine Learning Engineer positions at Fractal Analytics, all with a 72.5% match rate. This is a great sign that your skills, particularly in Python, Docker, and Linux, align well with the job requirements. 

These roles offer the opportunity to work on building and deploying machine learning models at scale, which can be a fantastic way to apply your skills in a real-world setting and continue to grow as a machine learning professional. The fact that all the top matches are from the same company suggests that Fractal Analytics m